In [0]:
# Read from Silver Delta table
df_silver = spark.read \
    .format("delta") \
    .table("workspace.insurance_claims.silver_claims_cleansed")

print("Records from Silver:", df_silver.count())
print("Columns:", len(df_silver.columns))

Records from Silver: 1000
Columns: 45


In [0]:
from pyspark.sql.functions import (
    count, sum as spark_sum, avg, max, min, round
)

# Monthly claims summary — your main fact table
df_claims_summary = df_silver.groupBy(
    "incident_year",
    "incident_month",
    "incident_type",
    "incident_severity",
    "claim_severity_band",
    "insured_sex",
    "age_band",
    "insured_education_level",
    "insured_occupation",
    "auto_make"
).agg(
    count("policy_number").alias("total_claims"),
    spark_sum("total_claim_amount").alias("total_claim_amount"),
    round(avg("total_claim_amount"), 2).alias("avg_claim_amount"),
    max("total_claim_amount").alias("max_claim_amount"),
    min("total_claim_amount").alias("min_claim_amount"),
    spark_sum("injury_claim").alias("total_injury_claim"),
    spark_sum("property_claim").alias("total_property_claim"),
    spark_sum("vehicle_claim").alias("total_vehicle_claim"),
    spark_sum("is_fraud").alias("total_fraud_claims"),
    round(avg("policy_age_days"), 0).alias("avg_policy_age_days")
)

print("Claims Summary rows:", df_claims_summary.count())
display(df_claims_summary)

Claims Summary rows: 992


incident_year,incident_month,incident_type,incident_severity,claim_severity_band,insured_sex,age_band,insured_education_level,insured_occupation,auto_make,total_claims,total_claim_amount,avg_claim_amount,max_claim_amount,min_claim_amount,total_injury_claim,total_property_claim,total_vehicle_claim,total_fraud_claims,avg_policy_age_days
2015,1,Single Vehicle Collision,MAJOR DAMAGE,High,MALE,Middle Aged,MD,craft-repair,Saab,1,71610,71610.0,71610,71610,6510,13020,52080,1,100.0
2015,1,Vehicle Theft,MINOR DAMAGE,Low,MALE,Middle Aged,MD,machine-op-inspct,Mercedes,1,5070,5070.0,5070,5070,780,780,3510,1,3130.0
2015,2,Multi-vehicle Collision,MINOR DAMAGE,Medium,FEMALE,Young Adult,PhD,sales,Dodge,1,34650,34650.0,34650,34650,7700,3850,23100,0,5282.0
2015,1,Single Vehicle Collision,MAJOR DAMAGE,High,FEMALE,Middle Aged,PhD,armed-forces,Chevrolet,2,105640,52820.0,63400,42240,14020,14020,77600,1,6931.0
2015,2,Vehicle Theft,MINOR DAMAGE,Low,MALE,Middle Aged,Associate,sales,Accura,1,6500,6500.0,6500,6500,1300,650,4550,0,256.0
2015,1,Multi-vehicle Collision,MAJOR DAMAGE,High,FEMALE,Young Adult,PhD,tech-support,Saab,1,64100,64100.0,64100,64100,6410,6410,51280,1,3004.0
2015,1,Multi-vehicle Collision,MINOR DAMAGE,High,MALE,Young Adult,PhD,prof-specialty,Nissan,1,78650,78650.0,78650,78650,21450,7150,50050,0,5336.0
2015,2,Multi-vehicle Collision,TOTAL LOSS,High,MALE,Young Adult,Associate,tech-support,Audi,1,51590,51590.0,51590,51590,9380,9380,32830,0,9155.0
2015,1,Single Vehicle Collision,TOTAL LOSS,Medium,FEMALE,Young Adult,PhD,other-service,Toyota,1,27700,27700.0,27700,27700,2770,2770,22160,0,6568.0
2015,1,Single Vehicle Collision,TOTAL LOSS,High,MALE,Middle Aged,PhD,priv-house-serv,Saab,1,42300,42300.0,42300,42300,4700,4700,32900,0,1260.0


In [0]:
from pyspark.sql.functions import (
    round, count, sum as spark_sum, avg, when, col
)

# Fraud analysis — Gold table 2
df_fraud_analysis = df_silver.groupBy(
    "incident_type",
    "incident_severity",
    "age_band",
    "insured_sex",
    "insured_occupation",
    "claim_severity_band",
    "authorities_contacted",
    "number_of_vehicles_involved"
).agg(
    count("policy_number").alias("total_claims"),
    spark_sum("is_fraud").alias("fraud_claims"),
    round(
        (spark_sum("is_fraud") / count("policy_number")) * 100, 2
    ).alias("fraud_percentage"),
    spark_sum("total_claim_amount").alias("total_amount"),
    round(avg("total_claim_amount"), 2).alias("avg_claim_amount")
)

# Add fraud risk label
df_fraud_analysis = df_fraud_analysis.withColumn(
    "fraud_risk_level",
    when(col("fraud_percentage") >= 75, "Critical")
    .when(col("fraud_percentage") >= 50, "High")
    .when(col("fraud_percentage") >= 25, "Medium")
    .otherwise("Low")
)

print("Fraud Analysis rows:", df_fraud_analysis.count())
display(df_fraud_analysis)

Fraud Analysis rows: 862


incident_type,incident_severity,age_band,insured_sex,insured_occupation,claim_severity_band,authorities_contacted,number_of_vehicles_involved,total_claims,fraud_claims,fraud_percentage,total_amount,avg_claim_amount,fraud_risk_level
Single Vehicle Collision,MAJOR DAMAGE,Middle Aged,MALE,craft-repair,High,Police,1,1,1,100.0,71610,71610.0,Critical
Vehicle Theft,MINOR DAMAGE,Middle Aged,MALE,machine-op-inspct,Low,Police,1,1,1,100.0,5070,5070.0,Critical
Multi-vehicle Collision,MINOR DAMAGE,Young Adult,FEMALE,sales,Medium,Police,3,1,0,0.0,34650,34650.0,Low
Single Vehicle Collision,MAJOR DAMAGE,Middle Aged,FEMALE,armed-forces,High,Police,1,1,1,100.0,63400,63400.0,Critical
Vehicle Theft,MINOR DAMAGE,Middle Aged,MALE,sales,Low,None,1,1,0,0.0,6500,6500.0,Low
Multi-vehicle Collision,MAJOR DAMAGE,Young Adult,FEMALE,tech-support,High,Fire,3,2,1,50.0,141980,70990.0,High
Multi-vehicle Collision,MINOR DAMAGE,Young Adult,MALE,prof-specialty,High,Police,3,1,0,0.0,78650,78650.0,Low
Multi-vehicle Collision,TOTAL LOSS,Young Adult,MALE,tech-support,High,Police,3,1,0,0.0,51590,51590.0,Low
Single Vehicle Collision,TOTAL LOSS,Young Adult,FEMALE,other-service,Medium,Police,1,1,0,0.0,27700,27700.0,Low
Single Vehicle Collision,TOTAL LOSS,Middle Aged,MALE,priv-house-serv,High,Other,1,3,0,0.0,162560,54186.67,Low


In [0]:
from pyspark.sql.functions import (
    count, sum as spark_sum, avg, 
    round, max, min, col, when
)

# Customer 360 — complete customer view
df_customer_360 = df_silver.groupBy(
    "insured_zip",
    "insured_sex",
    "age_band",
    "insured_education_level",
    "insured_occupation",
    "insured_relationship",
    "months_as_customer"
).agg(
    count("policy_number").alias("total_policies"),
    spark_sum("total_claim_amount").alias("lifetime_claim_amount"),
    round(avg("total_claim_amount"), 2).alias("avg_claim_amount"),
    max("total_claim_amount").alias("highest_claim"),
    min("total_claim_amount").alias("lowest_claim"),
    spark_sum("is_fraud").alias("fraud_count"),
    round(avg("policy_age_days"), 0).alias("avg_policy_age_days"),
    spark_sum("injury_claim").alias("total_injury_claims"),
    spark_sum("vehicle_claim").alias("total_vehicle_claims"),
    spark_sum("property_claim").alias("total_property_claims")
)

# Add customer risk label
df_customer_360 = df_customer_360.withColumn(
    "customer_risk_level",
    when(col("fraud_count") > 2, "High Risk")
    .when(col("fraud_count") > 0, "Medium Risk")
    .otherwise("Low Risk")
)

# Add customer loyalty band
df_customer_360 = df_customer_360.withColumn(
    "loyalty_band",
    when(col("months_as_customer") >= 300, "Platinum")
    .when(col("months_as_customer") >= 200, "Gold")
    .when(col("months_as_customer") >= 100, "Silver")
    .otherwise("Bronze")
)

print("Customer 360 rows:", df_customer_360.count())
display(df_customer_360)

Customer 360 rows: 1000


insured_zip,insured_sex,age_band,insured_education_level,insured_occupation,insured_relationship,months_as_customer,total_policies,lifetime_claim_amount,avg_claim_amount,highest_claim,lowest_claim,fraud_count,avg_policy_age_days,total_injury_claims,total_vehicle_claims,total_property_claims,customer_risk_level,loyalty_band
466132,MALE,Middle Aged,MD,craft-repair,husband,328,1,71610,71610.0,71610,71610,1,100.0,6510,52080,13020,Medium Risk,Platinum
468176,MALE,Middle Aged,MD,machine-op-inspct,other-relative,228,1,5070,5070.0,5070,5070,1,3130.0,780,3510,780,Medium Risk,Gold
430632,FEMALE,Young Adult,PhD,sales,own-child,134,1,34650,34650.0,34650,34650,0,5282.0,7700,23100,3850,Low Risk,Silver
608117,FEMALE,Middle Aged,PhD,armed-forces,unmarried,256,1,63400,63400.0,63400,63400,1,8996.0,6340,50720,6340,Medium Risk,Gold
610706,MALE,Middle Aged,Associate,sales,unmarried,228,1,6500,6500.0,6500,6500,0,256.0,1300,4550,650,Low Risk,Gold
478456,FEMALE,Young Adult,PhD,tech-support,unmarried,256,1,64100,64100.0,64100,64100,1,3004.0,6410,51280,6410,Medium Risk,Gold
441716,MALE,Young Adult,PhD,prof-specialty,husband,137,1,78650,78650.0,78650,78650,0,5336.0,21450,50050,7150,Low Risk,Silver
603195,MALE,Young Adult,Associate,tech-support,unmarried,165,1,51590,51590.0,51590,51590,0,9155.0,9380,32830,9380,Low Risk,Silver
601734,FEMALE,Young Adult,PhD,other-service,own-child,27,1,27700,27700.0,27700,27700,0,6568.0,2770,22160,2770,Low Risk,Bronze
600983,MALE,Middle Aged,PhD,priv-house-serv,wife,212,1,42300,42300.0,42300,42300,0,1260.0,4700,32900,4700,Low Risk,Gold


In [0]:
# Write Claims Summary
df_claims_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.insurance_claims.gold_claims_summary")

print("✅ Gold Claims Summary written!")

# Write Fraud Analysis
df_fraud_analysis.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.insurance_claims.gold_fraud_analysis")

print("✅ Gold Fraud Analysis written!")

# Write Customer 360
df_customer_360.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.insurance_claims.gold_customer_360")

print("✅ Gold Customer 360 written!")

print("\n🥇 All Gold tables written successfully!")

✅ Gold Claims Summary written!
✅ Gold Fraud Analysis written!
✅ Gold Customer 360 written!

🥇 All Gold tables written successfully!
